# 4.1 — K-Means & K-Means++

K-means turns unlabeled points into structure by replacing a cloud of data with a small set of representatives, then assigning each point to its nearest representative. In this lesson, you will build the objective, assignment step, centroid update, scaling checks, and k-means++ seeding rule from scratch in NumPy, so the clustering is an auditable optimization procedure rather than a mysterious scatterplot trick.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build k-means and k-means++ one idea at a time. Run each cell in order and read the printed intermediate values — every distance, assignment, centroid, and probability is exposed. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, distances, means, and reproducible sampling.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any randomized choices.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.


### 1. Unlabeled points and the clustering question

K-means starts with a matrix $X\in\mathbb{R}^{m\times n}$: rows are examples, columns are measured coordinates, and there are no labels. The question is not "what class is this point?" but "can a few representatives summarize the geometry?" Here we use six 2-D points that visually form two compact groups, but the algorithm will only see coordinates and distances.

In [ ]:
X_w = np.array([[1.0, 2.0], [1.5, 1.8], [0.8, 1.4],
                [4.8, 4.0], [5.2, 4.6], [4.4, 4.2]])  # six unlabeled 2-D points.
print("X shape:", X_w.shape)  # m=6 examples, n=2 coordinates.
print("first two points:\n", X_w[:2])  # inspect raw coordinates, not labels.
assert X_w.shape == (6, 2)  # shape bookkeeping prevents clustering bugs.

▶ What you'll see: a 6×2 matrix — the only input k-means receives.

In [ ]:
plt.figure(figsize=(4.4, 3.4))  # compact scatterplot of the unlabeled data.
plt.scatter(X_w[:, 0], X_w[:, 1], s=80, color="gray")  # draw points without cluster colors.
for i_w, (x_w, y_w) in enumerate(X_w):  # label each point by row index for debugging.
    plt.text(x_w + 0.04, y_w + 0.04, f"x{i_w}")
plt.title("1: unlabeled points before k-means")
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()

▶ What you'll see: two visible blobs, but no label column — clustering must infer structure from geometry.

*Why it's done this way: k-means is a compression lens. It assumes the data can be summarized by a few centers, so the raw object must be numeric coordinates where distance is meaningful.*

### 2. Squared Euclidean distance to candidate centers

The core measurement is squared Euclidean distance: $\lVert x-\mu\rVert_2^2=\sum_j(x_j-\mu_j)^2$. Squaring keeps distances nonnegative, makes larger misses count disproportionately, and gives a smooth objective whose centroid minimizer has a closed form. For one point, we compare two candidate centers and choose the nearer one.

In [ ]:
x_w = np.array([1.0, 2.0])       # one point to assign.
mu1_w = np.array([1.0, 1.5])     # nearby candidate representative.
mu2_w = np.array([4.5, 4.0])     # far candidate representative.
d1_w = float(np.sum((x_w - mu1_w) ** 2))  # (1-1)^2 + (2-1.5)^2.
d2_w = float(np.sum((x_w - mu2_w) ** 2))  # (1-4.5)^2 + (2-4)^2.
print("distance to mu1:", d1_w)
print("distance to mu2:", d2_w)
assert d1_w == 0.25 and d2_w == 16.25  # canonical arithmetic from the lesson content.

▶ What you'll see: the point is much closer to `mu1` (0.25) than to `mu2` (16.25).

In [ ]:
plt.figure(figsize=(4.4, 3.4))
plt.scatter([x_w[0]], [x_w[1]], s=100, color="black", label="point x")
plt.scatter([mu1_w[0], mu2_w[0]], [mu1_w[1], mu2_w[1]], s=130, marker="X", color=["seagreen", "crimson"], label="centers")
plt.plot([x_w[0], mu1_w[0]], [x_w[1], mu1_w[1]], color="seagreen")
plt.plot([x_w[0], mu2_w[0]], [x_w[1], mu2_w[1]], color="crimson")
plt.title("2: one point, two squared distances")
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.legend(); plt.show()

▶ What you'll see: the green segment is short and the red segment is long, matching the squared-distance numbers.

*Why it's done this way: nearest-center assignment turns a geometric question into a numeric decision; squaring makes the objective care much more about points that are far from their representative.*

### 3. Assignment: every point chooses its nearest center

With $K$ centers, assignment means computing every point-to-center distance and taking `argmin` across centers. The cluster label is not a truth label; it is the index of the representative that currently explains the point with the least squared error.

In [ ]:
centers_w = np.array([[1.0, 1.5], [4.5, 4.0]])  # two current representatives.
dists_w = np.sum((X_w[:, None, :] - centers_w[None, :, :]) ** 2, axis=2)  # shape (points, centers).
labels_w = np.argmin(dists_w, axis=1)  # nearest representative for each point.
print("distance matrix:\n", np.round(dists_w, 2))
print("labels:", labels_w)
assert labels_w.tolist() == [0, 0, 0, 1, 1, 1]  # the two visible blobs are separated.

▶ What you'll see: the first three rows have smaller distance to center 0, and the last three to center 1.

In [ ]:
plt.figure(figsize=(4.4, 3.4))
plt.scatter(X_w[:, 0], X_w[:, 1], c=labels_w, cmap="viridis", s=80)
plt.scatter(centers_w[:, 0], centers_w[:, 1], marker="X", s=180, color="red")
plt.title("3: assignment by nearest center")
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()

▶ What you'll see: the points are colored by their current nearest center, with red X markers showing the representatives.

*Why it's done this way: assignment is the discrete half of k-means. Once centers are fixed, the best label for each point is simply the center that contributes the smallest term to the objective.*

### 4. Centroid update: the mean minimizes within-cluster squared error

After assignment, k-means moves each representative to the average of the points assigned to it. This is not a heuristic: for squared error, the mean is the minimizer. In one dimension, minimizing $\sum_i(x_i-\mu)^2$ gives derivative $-2\sum_i(x_i-\mu)=0$, so $\mu=\frac{1}{m}\sum_i x_i$; the same argument applies coordinate-by-coordinate.

In [ ]:
new_centers_w = np.array([X_w[labels_w == k_w].mean(axis=0) for k_w in range(2)])  # average points in each cluster.
print("old centers:\n", centers_w)
print("updated centers:\n", np.round(new_centers_w, 3))
assert np.allclose(np.round(new_centers_w, 3), [[1.100, 1.733], [4.800, 4.267]])  # hand-checkable means.

▶ What you'll see: each center moves to the middle of its assigned blob.

In [ ]:
plt.figure(figsize=(4.4, 3.4))
plt.scatter(X_w[:, 0], X_w[:, 1], c=labels_w, cmap="viridis", s=80)
plt.scatter(centers_w[:, 0], centers_w[:, 1], marker="x", s=130, color="gray", label="old")
plt.scatter(new_centers_w[:, 0], new_centers_w[:, 1], marker="X", s=180, color="red", label="new")
for k_w in range(2):
    plt.arrow(centers_w[k_w, 0], centers_w[k_w, 1], new_centers_w[k_w, 0] - centers_w[k_w, 0], new_centers_w[k_w, 1] - centers_w[k_w, 1], color="black", head_width=0.07, length_includes_head=True)
plt.title("4: centroid update moves to means")
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.legend(); plt.show()

▶ What you'll see: arrows from old centers to updated means — the representatives move toward the middle of their assigned points.

*Why it's done this way: the mean is the squared-error-optimal representative for a fixed cluster, so k-means alternates between best labels for fixed centers and best centers for fixed labels.*

### 5. Objective J and alternating minimization

The k-means objective is $J=\sum_i\min_k\lVert x_i-\mu_k\rVert_2^2$. One iteration assigns points, updates centers, and should not increase $J$: assignment lowers or preserves the cost for fixed centers, and taking means lowers or preserves the cost for fixed labels. Repeating those two steps gives a local optimum.

In [ ]:
def kmeans_cost_w(X, C):  # compute J for a set of centers.
    D = np.sum((X[:, None, :] - C[None, :, :]) ** 2, axis=2)
    return float(np.sum(np.min(D, axis=1)))

cost_old_w = kmeans_cost_w(X_w, centers_w)  # objective before the centroid update.
cost_new_w = kmeans_cost_w(X_w, new_centers_w)  # objective after moving centers to means.
print("J before update:", round(cost_old_w, 3))
print("J after update:", round(cost_new_w, 3))
assert round(cost_old_w, 3) == 1.630 and round(cost_new_w, 3) == 0.953  # objective drops.

▶ What you'll see: the objective decreases from 1.63 to 0.953 after the mean update.

In [ ]:
C_loop_w = centers_w.copy()
costs_loop_w = []
for step_w in range(5):
    D_loop_w = np.sum((X_w[:, None, :] - C_loop_w[None, :, :]) ** 2, axis=2)
    y_loop_w = np.argmin(D_loop_w, axis=1)
    costs_loop_w.append(float(np.sum(D_loop_w[np.arange(len(X_w)), y_loop_w])))
    C_loop_w = np.array([X_w[y_loop_w == k_w].mean(axis=0) for k_w in range(2)])
print("costs:", np.round(costs_loop_w, 3))
assert costs_loop_w[-1] <= costs_loop_w[0]  # alternating steps do not climb here.

▶ What you'll see: the cost drops quickly and then stays flat once assignments stop changing.

In [ ]:
plt.figure(figsize=(4.4, 3.0))
plt.plot(costs_loop_w, marker="o", color="purple")
plt.title("5: k-means objective over iterations")
plt.xlabel("iteration"); plt.ylabel("J"); plt.show()

▶ What you'll see: a short descending curve — k-means converges when assignments and centers stabilize.

*Why it's done this way: k-means is not globally guaranteed, but each alternating subproblem is easy and locally improves the same objective, making the procedure simple, fast, and inspectable.*

### 6. K-means++: seed far-away points with probability

Bad initial centers can trap k-means in poor local optima. K-means++ fixes the first step: choose one center, compute each point's squared distance to its nearest chosen center, then sample the next center with probability proportional to that squared distance. Far unexplained regions get high probability, while already-covered points get near zero probability.

In [ ]:
first_center_w = X_w[0]  # pretend the first center was x0.
D2_w = np.sum((X_w - first_center_w) ** 2, axis=1)  # squared distance to nearest chosen center.
probs_w = D2_w / D2_w.sum()  # k-means++ sampling distribution for the next center.
print("D^2:", np.round(D2_w, 2))
print("probabilities:", np.round(probs_w, 3))
assert round(float(probs_w[:3].sum()), 3) == 0.012  # nearby first-blob points get little mass.
assert round(float(probs_w[3:].sum()), 3) == 0.988  # far second-blob points get most mass.

▶ What you'll see: almost all sampling probability goes to the far blob, because the first blob is already represented.

In [ ]:
plt.figure(figsize=(4.6, 3.2))
plt.bar([f"x{i_w}" for i_w in range(len(X_w))], probs_w, color="darkorange")
plt.title("6: k-means++ next-center probabilities")
plt.ylabel("probability ∝ D²"); plt.show()

▶ What you'll see: the last three bars dominate, making it likely that the second seed lands in the uncovered cluster.

*Why it's done this way: sampling by $D^2$ balances exploration and randomness — it strongly prefers regions with high current cost without deterministically picking only the single farthest point every time.*

### 7. Scaling and the "algorithm as lens" warning

K-means only knows the numeric scale you give it. If one feature is measured in huge units, that coordinate dominates squared distances and can decide clusters by accident. Standardizing features makes each coordinate contribute comparably before distances are computed.

In [ ]:
X_scale_w = np.array([[1.0, 10.0], [1.2, 12.0], [4.8, 11.0], [5.0, 13.0]])  # feature 0 separates groups; feature 1 has larger units.
C_scale_w = np.array([[1.1, 11.0], [4.9, 12.0]])  # intended representatives.
D_raw_w = np.sum((X_scale_w[:, None, :] - C_scale_w[None, :, :]) ** 2, axis=2)
raw_labels_w = np.argmin(D_raw_w, axis=1)
print("raw labels:", raw_labels_w)
print("raw distance first point:", np.round(D_raw_w[0], 2))

▶ What you'll see: raw squared distances mix units, so the larger-scale coordinate can dominate the decision.

In [ ]:
mean_scale_w = X_scale_w.mean(axis=0)
std_scale_w = X_scale_w.std(axis=0)
Z_scale_w = (X_scale_w - mean_scale_w) / std_scale_w  # standardize each coordinate.
Cz_scale_w = (C_scale_w - mean_scale_w) / std_scale_w
D_z_w = np.sum((Z_scale_w[:, None, :] - Cz_scale_w[None, :, :]) ** 2, axis=2)
z_labels_w = np.argmin(D_z_w, axis=1)
print("standardized labels:", z_labels_w)
print("standard deviations:", np.round(std_scale_w, 3))
assert np.allclose(np.round(Z_scale_w.mean(axis=0), 6), [0.0, 0.0])  # standardized coordinates are centered.

▶ What you'll see: after standardization, distances compare relative deviations instead of raw measurement units.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3.0))
ax[0].scatter(X_scale_w[:, 0], X_scale_w[:, 1], c=raw_labels_w, cmap="viridis", s=80)
ax[0].set_title("raw units")
ax[1].scatter(Z_scale_w[:, 0], Z_scale_w[:, 1], c=z_labels_w, cmap="viridis", s=80)
ax[1].set_title("standardized units")
for a_w in ax:
    a_w.set_xlabel("feature 0"); a_w.set_ylabel("feature 1")
plt.suptitle("7: scaling changes the k-means lens"); plt.show()

▶ What you'll see: the same rows can look different once features are put on comparable scales.

*Why it's done this way: the objective optimizes exactly the distances you define; preprocessing is therefore part of the model, not an optional cosmetic step.*

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, distances, means, random sampling, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib for scatterplots, bars, heatmaps, and convergence curves.
np.random.seed(0)  # make the examples reproducible across notebook runs.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.


## 🟢 Basics (warm-up)

### Basic 1 — Create an unlabeled point matrix

**Goal.** Store examples as rows and features as columns, because k-means only receives numeric coordinates. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[1.0, 2.0], [1.5, 1.8], [0.8, 1.4], [4.8, 4.0], [5.2, 4.6], [4.4, 4.2]])  # six 2-D unlabeled points.
print("shape:", X_b1.shape)  # inspect m examples by n coordinates.
print("first row:", X_b1[0])  # inspect one example before clustering.
assert X_b1.shape == (6, 2)  # concrete shape check.

▶ What you'll see: a 6×2 data matrix with no target labels.

In [ ]:
plt.figure(figsize=(4, 3))  # create a compact data plot.
plt.scatter(X_b1[:, 0], X_b1[:, 1], color="gray", s=80)  # draw points without cluster colors.
plt.title("Basic 1: unlabeled 2-D data")  # title the warm-up plot.
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()  # label and display.

▶ What you'll see: two natural-looking groups, even though the array has no labels.

👀 Takeaway: k-means starts from numeric geometry, so the row/column shape is the first modeling decision.

### Basic 2 — Compute one squared distance

**Goal.** Measure how far one point is from one center, because k-means uses squared Euclidean distance as its assignment cost. We build it in 2 steps.

In [ ]:
x_b2 = np.array([1.0, 2.0])  # choose one point.
mu_b2 = np.array([1.0, 1.5])  # choose one candidate center.
diff_b2 = x_b2 - mu_b2  # compute coordinate differences.
print("difference:", diff_b2)  # inspect the vector before squaring.

▶ What you'll see: only the second coordinate differs by 0.5.

In [ ]:
sqdist_b2 = float(np.sum(diff_b2 ** 2))  # sum squared coordinate differences.
print("squared distance:", sqdist_b2)  # inspect the scalar cost.
assert sqdist_b2 == 0.25  # verify the hand calculation.
plt.figure(figsize=(4, 3))
plt.bar(["dx²", "dy²"], diff_b2 ** 2, color="teal")
plt.title("Basic 2: squared-distance pieces"); plt.ylabel("squared contribution"); plt.show()

▶ What you'll see: the full distance cost comes from the second coordinate.

👀 Takeaway: squared distance is a sum of per-feature penalties, so each feature scale matters.

### Basic 3 — Compare one point to two centers

**Goal.** Choose the nearest representative for one point, because assignment is an argmin over candidate centers. We build it in 2 steps.

In [ ]:
x_b3 = np.array([1.0, 2.0])  # point to assign.
centers_b3 = np.array([[1.0, 1.5], [4.5, 4.0]])  # two candidate centers.
dists_b3 = np.sum((centers_b3 - x_b3) ** 2, axis=1)  # distance from x to each center.
print("distances:", dists_b3)  # inspect costs before choosing.
assert np.allclose(dists_b3, [0.25, 16.25])  # verify both distances.

▶ What you'll see: center 0 is much nearer than center 1.

In [ ]:
label_b3 = int(np.argmin(dists_b3))  # nearest-center index.
print("assigned center:", label_b3)  # inspect the cluster label.
plt.figure(figsize=(4, 3))
plt.bar(["center 0", "center 1"], dists_b3, color=["seagreen", "crimson"])
plt.title("Basic 3: nearest center wins"); plt.ylabel("squared distance"); plt.show()

▶ What you'll see: the smaller green bar explains why the assigned center is 0.

👀 Takeaway: a k-means label means "nearest current center," not a human-provided class.

### Basic 4 — Build a full distance matrix

**Goal.** Compute all point-to-center distances at once, because vectorization makes assignment inspectable and compact. We build it in 2 steps.

In [ ]:
X_b4 = np.array([[1.0, 2.0], [1.5, 1.8], [4.8, 4.0]])  # three points.
C_b4 = np.array([[1.0, 1.5], [4.5, 4.0]])  # two centers.
D_b4 = np.sum((X_b4[:, None, :] - C_b4[None, :, :]) ** 2, axis=2)  # broadcast to shape (3, 2).
print("D shape:", D_b4.shape)  # inspect point by center shape.
print(np.round(D_b4, 2))  # inspect numeric distances.

▶ What you'll see: each row contains the two candidate costs for one point.

In [ ]:
labels_b4 = np.argmin(D_b4, axis=1)  # choose the lowest-cost center per row.
print("labels:", labels_b4)  # inspect assignments.
assert labels_b4.tolist() == [0, 0, 1]  # verify expected nearest centers.
plt.figure(figsize=(4, 3))
plt.imshow(D_b4, cmap="magma", aspect="auto"); plt.colorbar(label="squared distance")
plt.title("Basic 4: distance matrix"); plt.xlabel("center"); plt.ylabel("point"); plt.show()

▶ What you'll see: dark cells mark the lower distance in each row.

👀 Takeaway: assignment is `argmin` across the center dimension of the distance matrix.

### Basic 5 — Update one centroid by averaging

**Goal.** Move a center to the mean of its assigned points, because the mean minimizes squared error inside a fixed cluster. We build it in 2 steps.

In [ ]:
cluster_b5 = np.array([[1.0, 2.0], [1.5, 1.8], [0.8, 1.4]])  # points assigned to one cluster.
centroid_b5 = cluster_b5.mean(axis=0)  # coordinate-wise average.
print("centroid:", np.round(centroid_b5, 3))  # inspect the new representative.
assert np.allclose(np.round(centroid_b5, 3), [1.100, 1.733])  # verify the mean.

▶ What you'll see: the centroid sits at the coordinate-wise average of the three points.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(cluster_b5[:, 0], cluster_b5[:, 1], s=80, color="steelblue", label="points")
plt.scatter([centroid_b5[0]], [centroid_b5[1]], marker="X", s=180, color="red", label="mean")
plt.title("Basic 5: centroid is the mean"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.legend(); plt.show()

▶ What you'll see: the red X is centered among the blue points.

👀 Takeaway: for squared distance, the average is the best representative of assigned points.

### Basic 6 — Compute the k-means objective J

**Goal.** Sum each point's nearest-center squared distance, because k-means optimizes this single scalar. We build it in 2 steps.

In [ ]:
X_b6 = np.array([[1.0, 2.0], [1.5, 1.8], [0.8, 1.4], [4.8, 4.0], [5.2, 4.6], [4.4, 4.2]])  # full toy data.
C_b6 = np.array([[1.0, 1.5], [4.5, 4.0]])  # current centers.
D_b6 = np.sum((X_b6[:, None, :] - C_b6[None, :, :]) ** 2, axis=2)  # all distances.
nearest_b6 = np.min(D_b6, axis=1)  # one cost per point.
print("nearest costs:", np.round(nearest_b6, 2))  # inspect objective terms.

▶ What you'll see: each point contributes only its distance to the closest center.

In [ ]:
J_b6 = float(np.sum(nearest_b6))  # k-means objective.
print("J:", round(J_b6, 3))  # inspect total fit cost.
assert round(J_b6, 3) == 1.630  # verify the objective for these centers.
plt.figure(figsize=(4, 3))
plt.bar(np.arange(len(nearest_b6)), nearest_b6, color="purple")
plt.title("Basic 6: objective terms"); plt.xlabel("point"); plt.ylabel("nearest squared distance"); plt.show()

▶ What you'll see: a bar for each point's contribution to J.

👀 Takeaway: k-means minimizes the sum of nearest-center squared errors.

### Basic 7 — Run one assign-update cycle

**Goal.** Combine assignment and centroid update once, because k-means alternates these two moves. We build it in 3 steps.

In [ ]:
X_b7 = np.array([[1.0, 2.0], [1.5, 1.8], [0.8, 1.4], [4.8, 4.0], [5.2, 4.6], [4.4, 4.2]])  # toy points.
C_old_b7 = np.array([[1.0, 1.5], [4.5, 4.0]])  # starting centers.
D_old_b7 = np.sum((X_b7[:, None, :] - C_old_b7[None, :, :]) ** 2, axis=2)  # distances.
labels_b7 = np.argmin(D_old_b7, axis=1)  # nearest centers.
print("labels:", labels_b7)  # inspect assignments before updating.

▶ What you'll see: the first three points choose center 0 and the last three choose center 1.

In [ ]:
C_new_b7 = np.array([X_b7[labels_b7 == k_b7].mean(axis=0) for k_b7 in range(2)])  # update centers to cluster means.
print("new centers:\n", np.round(C_new_b7, 3))  # inspect updated centers.
assert np.allclose(np.round(C_new_b7, 3), [[1.100, 1.733], [4.800, 4.267]])  # verify update.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_b7[:, 0], X_b7[:, 1], c=labels_b7, cmap="viridis", s=80)
plt.scatter(C_old_b7[:, 0], C_old_b7[:, 1], marker="x", s=120, color="gray", label="old")
plt.scatter(C_new_b7[:, 0], C_new_b7[:, 1], marker="X", s=160, color="red", label="new")
plt.title("Basic 7: one k-means cycle"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.legend(); plt.show()

▶ What you'll see: old centers move into the middle of their assigned colored points.

👀 Takeaway: k-means improves centers only after labels have been recomputed from distances.

### Basic 8 — Detect convergence by unchanged labels

**Goal.** Stop when assignments no longer change, because additional iterations then repeat the same centers. We build it in 3 steps.

In [ ]:
X_b8 = np.array([[1.0, 2.0], [1.5, 1.8], [0.8, 1.4], [4.8, 4.0], [5.2, 4.6], [4.4, 4.2]])  # toy points.
C_b8 = np.array([[1.0, 1.5], [4.5, 4.0]])  # initial centers.
prev_b8 = None  # no previous labels yet.
history_b8 = []  # store objective values.
print("starting centers:\n", C_b8)  # inspect initial state.

▶ What you'll see: the loop starts from the same two representatives used above.

In [ ]:
for step_b8 in range(10):  # enough iterations for this tiny dataset.
    D_b8 = np.sum((X_b8[:, None, :] - C_b8[None, :, :]) ** 2, axis=2)  # distance matrix.
    labels_b8 = np.argmin(D_b8, axis=1)  # assignment step.
    history_b8.append(float(np.sum(D_b8[np.arange(len(X_b8)), labels_b8])))  # record J.
    if prev_b8 is not None and np.array_equal(labels_b8, prev_b8):  # convergence check.
        break
    prev_b8 = labels_b8.copy()  # remember labels.
    C_b8 = np.array([X_b8[labels_b8 == k_b8].mean(axis=0) for k_b8 in range(2)])  # update step.
print("iterations run:", step_b8 + 1)
print("cost history:", np.round(history_b8, 3))
assert step_b8 + 1 <= 3  # this simple case stabilizes quickly.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(history_b8, marker="o", color="darkorange")
plt.title("Basic 8: convergence by stable labels"); plt.xlabel("iteration"); plt.ylabel("J"); plt.show()

▶ What you'll see: the objective drops and then flattens when assignments stop changing.

👀 Takeaway: convergence means the assign-update loop has reached a fixed point, usually a local optimum.

### Basic 9 — Inspect k-means++ probabilities

**Goal.** Compute the distribution for the second seed, because k-means++ chooses new centers in proportion to current squared distance. We build it in 2 steps.

In [ ]:
X_b9 = np.array([[1.0, 2.0], [1.5, 1.8], [0.8, 1.4], [4.8, 4.0], [5.2, 4.6], [4.4, 4.2]])  # toy points.
first_b9 = X_b9[0]  # first chosen seed.
D2_b9 = np.sum((X_b9 - first_b9) ** 2, axis=1)  # squared distance to nearest chosen seed.
probs_b9 = D2_b9 / D2_b9.sum()  # normalize to probabilities.
print("D^2:", np.round(D2_b9, 2))
print("probabilities:", np.round(probs_b9, 3))
assert round(float(probs_b9.sum()), 6) == 1.0  # probabilities normalize.

▶ What you'll see: far points receive much larger probability than nearby points.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(len(X_b9)), probs_b9, color="goldenrod")
plt.title("Basic 9: k-means++ sampling mass"); plt.xlabel("point index"); plt.ylabel("probability"); plt.show()

▶ What you'll see: most of the next-center mass sits on the second blob.

👀 Takeaway: k-means++ uses distance-weighted randomness to spread initial centers across unexplained regions.

### Basic 10 — Standardize features before distances

**Goal.** Put features on comparable scales, because squared distances inherit the units of the input columns. We build it in 2 steps.

In [ ]:
X_b10 = np.array([[1.0, 10.0], [1.2, 12.0], [4.8, 11.0], [5.0, 13.0]])  # two features with different scales.
mean_b10 = X_b10.mean(axis=0)  # feature means.
std_b10 = X_b10.std(axis=0)  # feature standard deviations.
Z_b10 = (X_b10 - mean_b10) / std_b10  # standardized data.
print("mean:", mean_b10, "std:", np.round(std_b10, 3))
assert np.allclose(np.round(Z_b10.mean(axis=0), 6), [0.0, 0.0])  # standardized means are zero.

▶ What you'll see: each column has its own center and scale.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(Z_b10[:, 0], Z_b10[:, 1], s=90, color="teal")
plt.axhline(0, color="gray", linewidth=0.7); plt.axvline(0, color="gray", linewidth=0.7)
plt.title("Basic 10: standardized coordinates"); plt.xlabel("z feature 0"); plt.ylabel("z feature 1"); plt.show()

▶ What you'll see: standardized values are centered near 0 on both axes.

👀 Takeaway: scaling is part of the k-means model because it changes the distances being optimized.

## 🟡 Easy

### Easy 1 — Fit k-means from scratch

**Goal.** Implement the full assign-update loop, because k-means is just repeated nearest-center assignment plus centroid recomputation. We build it in 4 steps.

In [ ]:
X_e1 = np.array([[1.0, 2.0], [1.5, 1.8], [0.8, 1.4], [4.8, 4.0], [5.2, 4.6], [4.4, 4.2]])  # toy data.
C_e1 = np.array([[1.0, 1.5], [4.5, 4.0]])  # deterministic starting centers.
costs_e1 = []  # objective history.
print("initial centers:\n", C_e1)  # inspect start.

▶ What you'll see: two initial representatives, one near each blob.

In [ ]:
for step_e1 in range(6):  # run several k-means iterations.
    D_e1 = np.sum((X_e1[:, None, :] - C_e1[None, :, :]) ** 2, axis=2)  # distances.
    labels_e1 = np.argmin(D_e1, axis=1)  # assignment.
    costs_e1.append(float(np.sum(D_e1[np.arange(len(X_e1)), labels_e1])))  # objective.
    C_next_e1 = np.array([X_e1[labels_e1 == k_e1].mean(axis=0) for k_e1 in range(2)])  # centroid update.
    if np.allclose(C_next_e1, C_e1):  # stop if centers no longer move.
        break
    C_e1 = C_next_e1  # continue from updated centers.
print("final centers:\n", np.round(C_e1, 3))
print("costs:", np.round(costs_e1, 3))

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
assert np.allclose(np.round(C_e1, 3), [[1.100, 1.733], [4.800, 4.267]])  # final representatives.
assert costs_e1[-1] <= costs_e1[0]  # fit improved.
plt.figure(figsize=(4, 3))
plt.plot(costs_e1, marker="o", color="purple")
plt.title("Easy 1: k-means convergence"); plt.xlabel("iteration"); plt.ylabel("J"); plt.show()

▶ What you'll see: the cost falls to the stable objective for the two clusters.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_e1[:, 0], X_e1[:, 1], c=labels_e1, cmap="viridis", s=80)
plt.scatter(C_e1[:, 0], C_e1[:, 1], marker="X", s=180, color="red")
plt.title("Easy 1: learned clusters"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()

▶ What you'll see: learned red centers sit in the middle of the colored groups.

👀 Takeaway: k-means is an alternating minimization loop over one objective, not a black-box classifier.

### Easy 2 — Compare random seeding with k-means++ seeding

**Goal.** Show why initialization matters, because k-means can converge to different local optima from different starting centers. We build it in 4 steps.

In [ ]:
X_e2 = np.array([[0.0, 0.0], [0.2, 0.1], [-0.1, 0.2], [4.0, 4.0], [4.2, 3.9], [3.8, 4.1], [8.0, 0.0], [8.2, 0.1], [7.8, -0.1]])  # three blobs.
bad_C_e2 = X_e2[[0, 1, 3]].copy()  # two seeds accidentally start in the left blob.
pp_C_e2 = X_e2[[0, 3, 6]].copy()  # spread-out k-means++-style seeds.
print("bad seed indices: [0, 1, 3]")
print("spread seed indices: [0, 3, 6]")

▶ What you'll see: the bad start wastes two centers near the same region.

In [ ]:
C_bad_e2 = bad_C_e2.copy()
for step_bad_e2 in range(8):
    D_bad_e2 = np.sum((X_e2[:, None, :] - C_bad_e2[None, :, :]) ** 2, axis=2)
    y_bad_e2 = np.argmin(D_bad_e2, axis=1)
    C_bad_e2 = np.array([X_e2[y_bad_e2 == k_e2].mean(axis=0) if np.any(y_bad_e2 == k_e2) else C_bad_e2[k_e2] for k_e2 in range(3)])
J_bad_e2 = float(np.sum(np.min(np.sum((X_e2[:, None, :] - C_bad_e2[None, :, :]) ** 2, axis=2), axis=1)))
print("bad-start J:", round(J_bad_e2, 3))

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
C_pp_e2 = pp_C_e2.copy()
for step_pp_e2 in range(8):
    D_pp_e2 = np.sum((X_e2[:, None, :] - C_pp_e2[None, :, :]) ** 2, axis=2)
    y_pp_e2 = np.argmin(D_pp_e2, axis=1)
    C_pp_e2 = np.array([X_e2[y_pp_e2 == k_e2].mean(axis=0) for k_e2 in range(3)])
J_pp_e2 = float(np.sum(np.min(np.sum((X_e2[:, None, :] - C_pp_e2[None, :, :]) ** 2, axis=2), axis=1)))
print("spread-start J:", round(J_pp_e2, 3))
assert J_pp_e2 <= J_bad_e2  # spread seeds should be no worse here.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].scatter(X_e2[:, 0], X_e2[:, 1], c=y_bad_e2, cmap="viridis", s=70); ax[0].scatter(C_bad_e2[:, 0], C_bad_e2[:, 1], marker="X", s=150, color="red"); ax[0].set_title(f"bad start J={J_bad_e2:.2f}")
ax[1].scatter(X_e2[:, 0], X_e2[:, 1], c=y_pp_e2, cmap="viridis", s=70); ax[1].scatter(C_pp_e2[:, 0], C_pp_e2[:, 1], marker="X", s=150, color="red"); ax[1].set_title(f"spread start J={J_pp_e2:.2f}")
for a_e2 in ax:
    a_e2.set_xlabel("feature 0"); a_e2.set_ylabel("feature 1")
plt.suptitle("Easy 2: initialization changes the local optimum"); plt.show()

▶ What you'll see: spread-out seeds cover the three blobs more cleanly than a duplicated start.

👀 Takeaway: k-means++ improves the starting lens by making duplicated seeds unlikely.

### Easy 3 — Choose k with an elbow curve

**Goal.** Sweep the number of clusters and plot the objective, because larger k always lowers training cost and the elbow helps reveal diminishing returns. We build it in 3 steps.

In [ ]:
X_e3 = np.array([[0.0, 0.0], [0.2, 0.1], [-0.1, 0.2], [4.0, 4.0], [4.2, 3.9], [3.8, 4.1], [8.0, 0.0], [8.2, 0.1], [7.8, -0.1]])  # three-blob data.
ks_e3 = np.array([1, 2, 3, 4, 5])  # candidate cluster counts.
J_e3 = []  # objective per k.
print("k grid:", ks_e3)

▶ What you'll see: the sweep tests increasingly flexible summaries.

In [ ]:
for k_e3 in ks_e3:
    C_e3 = X_e3[np.linspace(0, len(X_e3) - 1, k_e3, dtype=int)].copy()  # deterministic spread seeds.
    for step_e3 in range(10):
        D_e3 = np.sum((X_e3[:, None, :] - C_e3[None, :, :]) ** 2, axis=2)
        y_e3 = np.argmin(D_e3, axis=1)
        C_e3 = np.array([X_e3[y_e3 == c_e3].mean(axis=0) if np.any(y_e3 == c_e3) else C_e3[c_e3] for c_e3 in range(k_e3)])
    J_e3.append(float(np.sum(np.min(np.sum((X_e3[:, None, :] - C_e3[None, :, :]) ** 2, axis=2), axis=1))))
print("J by k:", np.round(J_e3, 3))
assert J_e3[2] < J_e3[1] and J_e3[3] <= J_e3[2]  # more clusters do not increase training fit cost.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(ks_e3, J_e3, marker="o", color="navy")
plt.axvline(3, color="crimson", linestyle="--", label="visible elbow")
plt.title("Easy 3: elbow curve"); plt.xlabel("k"); plt.ylabel("J"); plt.xticks(ks_e3); plt.legend(); plt.show()

▶ What you'll see: the largest drop happens before k=3, then extra clusters mostly refine small details.

👀 Takeaway: the elbow is a diagnostic for useful compression, not proof of a true hidden label count.

### Easy 4 — See how scaling can flip assignments

**Goal.** Compare raw and standardized distances, because unscaled features silently define what "close" means. We build it in 3 steps.

In [ ]:
X_e4 = np.array([[0.0, 0.0], [0.0, 100.0], [10.0, 45.0], [10.0, 55.0]])  # one feature has much larger numeric range.
C_e4 = np.array([[0.0, 50.0], [10.0, 50.0]])  # centers separated on feature 0.
raw_D_e4 = np.sum((X_e4[:, None, :] - C_e4[None, :, :]) ** 2, axis=2)
raw_y_e4 = np.argmin(raw_D_e4, axis=1)
print("raw labels:", raw_y_e4)

▶ What you'll see: raw units strongly emphasize the large-scale second coordinate.

In [ ]:
Z_e4 = (X_e4 - X_e4.mean(axis=0)) / X_e4.std(axis=0)  # standardize data.
CZ_e4 = (C_e4 - X_e4.mean(axis=0)) / X_e4.std(axis=0)  # standardize centers with same transform.
z_D_e4 = np.sum((Z_e4[:, None, :] - CZ_e4[None, :, :]) ** 2, axis=2)
z_y_e4 = np.argmin(z_D_e4, axis=1)
print("standardized labels:", z_y_e4)
assert raw_D_e4.shape == z_D_e4.shape == (4, 2)  # both are point-by-center matrices.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].scatter(X_e4[:, 0], X_e4[:, 1], c=raw_y_e4, cmap="viridis", s=80); ax[0].scatter(C_e4[:, 0], C_e4[:, 1], marker="X", s=150, color="red"); ax[0].set_title("raw distances")
ax[1].scatter(Z_e4[:, 0], Z_e4[:, 1], c=z_y_e4, cmap="viridis", s=80); ax[1].scatter(CZ_e4[:, 0], CZ_e4[:, 1], marker="X", s=150, color="red"); ax[1].set_title("standardized distances")
for a_e4 in ax:
    a_e4.set_xlabel("feature 0"); a_e4.set_ylabel("feature 1")
plt.suptitle("Easy 4: scaling changes assignments"); plt.show()

▶ What you'll see: assignments are based on the distance geometry after preprocessing.

👀 Takeaway: feature scaling is a modeling choice because it changes the k-means objective itself.

### Easy 5 — Evaluate clustering stability across seeds

**Goal.** Rerun k-means with several initializations, because unstable results warn that the discovered structure may be fragile. We build it in 4 steps.

In [ ]:
X_e5 = np.array([[0.0, 0.0], [0.2, 0.1], [-0.1, 0.2], [4.0, 4.0], [4.2, 3.9], [3.8, 4.1], [8.0, 0.0], [8.2, 0.1], [7.8, -0.1]])  # three compact blobs.
seeds_e5 = np.arange(6)  # several random initializations.
final_J_e5 = []  # store final objective per seed.
print("seeds:", seeds_e5)

▶ What you'll see: stability is checked by repeating the same algorithm from different starts.

In [ ]:
for seed_e5 in seeds_e5:
    rng_e5 = np.random.default_rng(seed_e5)
    C_e5 = X_e5[rng_e5.choice(len(X_e5), size=3, replace=False)].copy()
    for step_e5 in range(10):
        D_e5 = np.sum((X_e5[:, None, :] - C_e5[None, :, :]) ** 2, axis=2)
        y_e5 = np.argmin(D_e5, axis=1)
        C_e5 = np.array([X_e5[y_e5 == k_e5].mean(axis=0) if np.any(y_e5 == k_e5) else C_e5[k_e5] for k_e5 in range(3)])
    final_J_e5.append(float(np.sum(np.min(np.sum((X_e5[:, None, :] - C_e5[None, :, :]) ** 2, axis=2), axis=1))))
print("final J values:", np.round(final_J_e5, 3))

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
spread_e5 = float(np.max(final_J_e5) - np.min(final_J_e5))  # instability measure.
print("objective spread:", round(spread_e5, 3))
assert spread_e5 >= 0.0  # spread is nonnegative by definition.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar([str(s_e5) for s_e5 in seeds_e5], final_J_e5, color="slateblue")
plt.title("Easy 5: final objective by seed"); plt.xlabel("seed"); plt.ylabel("final J"); plt.show()

▶ What you'll see: some starts may land at the same cost, while worse starts reveal local-optimum risk.

👀 Takeaway: rerunning with multiple seeds is a practical stability check for unsupervised claims.

## 🔴 Advanced

### Advanced 1 — Implement k-means++ initialization

**Goal.** Build the full k-means++ seeding procedure, because good starts reduce duplicated centers and poor local optima. We build it in 4 steps.

In [ ]:
X_a1 = np.array([[0.0, 0.0], [0.2, 0.1], [-0.1, 0.2], [4.0, 4.0], [4.2, 3.9], [3.8, 4.1], [8.0, 0.0], [8.2, 0.1], [7.8, -0.1]])  # three blobs.
rng_a1 = np.random.default_rng(1)  # local generator for reproducible seeding.
centers_a1 = [X_a1[0]]  # choose the first center deterministically for auditability.
print("first center:", centers_a1[0])

▶ What you'll see: seeding begins from one known point so the probability calculations can be inspected.

In [ ]:
while len(centers_a1) < 3:
    C_now_a1 = np.array(centers_a1)
    D_now_a1 = np.sum((X_a1[:, None, :] - C_now_a1[None, :, :]) ** 2, axis=2)
    nearest_a1 = np.min(D_now_a1, axis=1)
    probs_a1 = nearest_a1 / nearest_a1.sum()
    next_idx_a1 = int(rng_a1.choice(len(X_a1), p=probs_a1))
    centers_a1.append(X_a1[next_idx_a1])
    print("chosen index:", next_idx_a1, "prob:", round(float(probs_a1[next_idx_a1]), 3))
C_a1 = np.array(centers_a1)
print("initial centers:\n", C_a1)
assert C_a1.shape == (3, 2)  # three 2-D seeds.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
for step_a1 in range(8):
    D_a1 = np.sum((X_a1[:, None, :] - C_a1[None, :, :]) ** 2, axis=2)
    y_a1 = np.argmin(D_a1, axis=1)
    C_a1 = np.array([X_a1[y_a1 == k_a1].mean(axis=0) for k_a1 in range(3)])
J_a1 = float(np.sum(np.min(np.sum((X_a1[:, None, :] - C_a1[None, :, :]) ** 2, axis=2), axis=1)))
print("final J:", round(J_a1, 3))
assert J_a1 < 1.0  # compact blobs should be fit tightly.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(X_a1[:, 0], X_a1[:, 1], c=y_a1, cmap="viridis", s=80)
plt.scatter(C_a1[:, 0], C_a1[:, 1], marker="X", s=180, color="red")
plt.title("Advanced 1: k-means++ seeds then fit"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()

▶ What you'll see: k-means++ seeds lead to one center per visible blob.

👀 Takeaway: k-means++ is a probabilistic preprocessing step that improves the same k-means objective.

### Advanced 2 — Use inertia and silhouette-style separation

**Goal.** Combine within-cluster cost with a separation diagnostic, because a low objective alone can still overfit by using too many clusters. We build it in 4 steps.

In [ ]:
X_a2 = np.array([[0.0, 0.0], [0.2, 0.1], [-0.1, 0.2], [4.0, 4.0], [4.2, 3.9], [3.8, 4.1], [8.0, 0.0], [8.2, 0.1], [7.8, -0.1]])  # three blobs.
C_a2 = np.array([[0.0, 0.0], [4.0, 4.0], [8.0, 0.0]])  # good initial centers.
for step_a2 in range(5):
    D_a2 = np.sum((X_a2[:, None, :] - C_a2[None, :, :]) ** 2, axis=2)
    y_a2 = np.argmin(D_a2, axis=1)
    C_a2 = np.array([X_a2[y_a2 == k_a2].mean(axis=0) for k_a2 in range(3)])
print("centers:\n", np.round(C_a2, 3))

▶ What you'll see: centers settle near the three blob means.

In [ ]:
pair_D_a2 = np.sqrt(np.sum((X_a2[:, None, :] - X_a2[None, :, :]) ** 2, axis=2))  # all point-to-point distances.
sil_a2 = []  # approximate silhouette per point.
for i_a2 in range(len(X_a2)):
    same_a2 = y_a2 == y_a2[i_a2]
    other_vals_a2 = []
    a_val_a2 = float(np.mean(pair_D_a2[i_a2, same_a2 & (np.arange(len(X_a2)) != i_a2)]))
    for k_a2 in range(3):
        if k_a2 != y_a2[i_a2]:
            other_vals_a2.append(float(np.mean(pair_D_a2[i_a2, y_a2 == k_a2])))
    b_val_a2 = min(other_vals_a2)
    sil_a2.append((b_val_a2 - a_val_a2) / max(a_val_a2, b_val_a2))
sil_a2 = np.array(sil_a2)
print("mean silhouette-style score:", round(float(np.mean(sil_a2)), 3))
assert np.mean(sil_a2) > 0.8  # blobs are well separated.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
inertia_a2 = float(np.sum(np.min(np.sum((X_a2[:, None, :] - C_a2[None, :, :]) ** 2, axis=2), axis=1)))
print("inertia J:", round(inertia_a2, 3))
print("cluster sizes:", [int(np.sum(y_a2 == k_a2)) for k_a2 in range(3)])

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(len(sil_a2)), sil_a2, color="seagreen")
plt.axhline(np.mean(sil_a2), color="black", linestyle="--", label="mean")
plt.title("Advanced 2: separation by point"); plt.xlabel("point"); plt.ylabel("silhouette-style score"); plt.legend(); plt.show()

▶ What you'll see: most points have high separation scores, meaning their own cluster is closer than neighboring clusters.

👀 Takeaway: unsupervised evaluation should inspect both fit and separation, not training objective alone.

### Advanced 3 — Mini-batch k-means updates

**Goal.** Update centers from small batches, because large datasets often cannot afford full passes over every point for every update. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(3)  # reproducible synthetic data.
blob1_a3 = rng_a3.normal(loc=[0.0, 0.0], scale=0.25, size=(30, 2))
blob2_a3 = rng_a3.normal(loc=[4.0, 4.0], scale=0.25, size=(30, 2))
X_a3 = np.vstack([blob1_a3, blob2_a3])  # sixty points.
C_a3 = X_a3[[0, 30]].copy()  # start one center in each blob.
counts_a3 = np.zeros(2)  # number of assigned mini-batch updates per center.
print("X shape:", X_a3.shape)

▶ What you'll see: a larger synthetic dataset still represented as an m×2 matrix.

In [ ]:
trace_a3 = []  # store occasional objective values.
for step_a3 in range(80):
    batch_idx_a3 = rng_a3.choice(len(X_a3), size=8, replace=False)  # small batch.
    batch_a3 = X_a3[batch_idx_a3]
    D_batch_a3 = np.sum((batch_a3[:, None, :] - C_a3[None, :, :]) ** 2, axis=2)
    y_batch_a3 = np.argmin(D_batch_a3, axis=1)
    for point_a3, label_a3 in zip(batch_a3, y_batch_a3):
        counts_a3[label_a3] += 1
        rate_a3 = 1.0 / counts_a3[label_a3]  # running-mean learning rate for that center.
        C_a3[label_a3] = (1 - rate_a3) * C_a3[label_a3] + rate_a3 * point_a3
    if step_a3 % 5 == 0:
        D_full_a3 = np.sum((X_a3[:, None, :] - C_a3[None, :, :]) ** 2, axis=2)
        trace_a3.append(float(np.sum(np.min(D_full_a3, axis=1))))
print("final centers:\n", np.round(C_a3, 3))
assert trace_a3[-1] < trace_a3[0]  # mini-batch updates improved fit.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
print("objective start -> end:", round(trace_a3[0], 3), "->", round(trace_a3[-1], 3))
print("center update counts:", counts_a3.astype(int))

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(np.arange(len(trace_a3)) * 5, trace_a3, marker="o", color="purple")
plt.title("Advanced 3: mini-batch objective trace"); plt.xlabel("mini-batch step"); plt.ylabel("full-data J"); plt.show()

▶ What you'll see: noisy but downward progress as small batches move centers toward blob means.

👀 Takeaway: mini-batch k-means trades exact full updates for cheaper approximate progress.

### Advanced 4 — Spot non-spherical cluster failure

**Goal.** Show a limitation of k-means, because nearest-centroid squared distance prefers round, similarly sized clusters. We build it in 4 steps.

In [ ]:
rng_a4 = np.random.default_rng(4)  # reproducible elongated data.
line_a4 = np.column_stack([np.linspace(0, 6, 40), 0.25 * rng_a4.normal(size=40)])  # elongated horizontal group.
blob_a4 = rng_a4.normal(loc=[3.0, 3.0], scale=[0.35, 0.35], size=(20, 2))  # compact blob.
X_a4 = np.vstack([line_a4, blob_a4])
C_a4 = np.array([[1.0, 0.0], [5.0, 0.0], [3.0, 3.0]])  # k=3 centers can split the long line.
print("X shape:", X_a4.shape)

▶ What you'll see: the dataset contains one stretched structure and one compact structure.

In [ ]:
for step_a4 in range(12):
    D_a4 = np.sum((X_a4[:, None, :] - C_a4[None, :, :]) ** 2, axis=2)
    y_a4 = np.argmin(D_a4, axis=1)
    C_a4 = np.array([X_a4[y_a4 == k_a4].mean(axis=0) if np.any(y_a4 == k_a4) else C_a4[k_a4] for k_a4 in range(3)])
counts_a4 = np.array([np.sum(y_a4 == k_a4) for k_a4 in range(3)])
print("cluster sizes:", counts_a4)
assert counts_a4.sum() == len(X_a4)  # every point is assigned.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
J_a4 = float(np.sum(np.min(np.sum((X_a4[:, None, :] - C_a4[None, :, :]) ** 2, axis=2), axis=1)))
print("final J:", round(J_a4, 3))
print("centers:\n", np.round(C_a4, 3))

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(X_a4[:, 0], X_a4[:, 1], c=y_a4, cmap="viridis", s=35)
plt.scatter(C_a4[:, 0], C_a4[:, 1], marker="X", s=170, color="red")
plt.title("Advanced 4: k-means can split elongated structure"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()

▶ What you'll see: the long horizontal structure is divided by centroid geometry even if you might think of it as one pattern.

👀 Takeaway: k-means finds Voronoi-style round partitions, so non-spherical structure needs a different lens or features.

### Advanced 5 — Compress data and measure reconstruction error

**Goal.** Treat cluster centers as a lossy codebook, because k-means also compresses each point into a center index plus a representative. We build it in 4 steps.

In [ ]:
X_a5 = np.array([[1.0, 2.0], [1.5, 1.8], [0.8, 1.4], [4.8, 4.0], [5.2, 4.6], [4.4, 4.2]])  # toy data.
C_a5 = np.array([[1.0, 1.5], [4.5, 4.0]])  # starting codebook.
for step_a5 in range(5):
    D_a5 = np.sum((X_a5[:, None, :] - C_a5[None, :, :]) ** 2, axis=2)
    y_a5 = np.argmin(D_a5, axis=1)
    C_a5 = np.array([X_a5[y_a5 == k_a5].mean(axis=0) for k_a5 in range(2)])
print("codebook centers:\n", np.round(C_a5, 3))

▶ What you'll see: the codebook contains one representative per learned cluster.

In [ ]:
X_recon_a5 = C_a5[y_a5]  # replace each point with its assigned center.
errors_a5 = np.sum((X_a5 - X_recon_a5) ** 2, axis=1)  # reconstruction error per point.
print("reconstruction errors:", np.round(errors_a5, 3))
print("total error:", round(float(errors_a5.sum()), 3))
assert round(float(errors_a5.sum()), 3) == 0.953  # same as the k-means objective at convergence.

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
compression_ratio_a5 = X_a5.size / (C_a5.size + y_a5.size)  # raw numbers divided by codebook+labels numbers.
print("raw numbers:", X_a5.size, "compressed numbers:", C_a5.size + y_a5.size)
print("compression ratio:", round(float(compression_ratio_a5), 3))

▶ What you'll see: the printed intermediate values expose this step of the calculation so you can audit it before continuing.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_a5[:, 0], X_a5[:, 1], color="gray", s=70, label="original")
plt.scatter(X_recon_a5[:, 0], X_recon_a5[:, 1], marker="X", color="red", s=130, label="reconstruction")
for i_a5 in range(len(X_a5)):
    plt.plot([X_a5[i_a5, 0], X_recon_a5[i_a5, 0]], [X_a5[i_a5, 1], X_recon_a5[i_a5, 1]], color="black", linewidth=0.7)
plt.title("Advanced 5: k-means as lossy compression"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.legend(); plt.show()

▶ What you'll see: each original point is replaced by its cluster center, and the connecting lines are the reconstruction errors.

👀 Takeaway: k-means minimizes the squared reconstruction error of replacing each point by a small codebook of centers.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

k-means & k-means++ turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise.

Distances, averages, and nearest-neighbor reasoning become an assign/update loop. The method compresses points into representative centroids, so scaling and initialization are part of the model, not bookkeeping.

Save a copy to Drive to edit. This notebook is deterministic, CPU-only, and uses only bundled scikit-learn data.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from collections import deque
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import AffinityPropagation
from sklearn.cluster import Birch
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
from sklearn.cluster import OPTICS
from sklearn.cluster import estimate_bandwidth
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

SEED = 7
rng = np.random.default_rng(SEED)

def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty. Returns [(name, X, y_true, k), ...].

    y_true is the generating label (for ARI scoring only — clustering does not see it).
    Rungs: hand points -> clean blobs -> anisotropic/overlap -> real Iris -> real digits(4-class).
    """
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs



def project_2d(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=SEED).fit_transform(X)


def ari_score(y_true, labels):
    return float(adjusted_rand_score(y_true, labels))


def safe_silhouette(X, labels):
    labels = np.asarray(labels)
    keep = labels != -1
    unique = np.unique(labels[keep])
    if keep.sum() < 3:
        return np.nan
    if unique.size < 2:
        return np.nan
    if unique.size >= keep.sum():
        return np.nan
    return float(silhouette_score(X[keep], labels[keep]))


def preview_ladder(rungs):
    rows = []
    for idx, item in enumerate(rungs, start=1):
        name, X, y_true, k = item
        rows.append((idx, name, X.shape, int(np.unique(y_true).size), k, np.round(X[:3], 3).tolist()))
    return rows


def plot_cluster_panels(results, title, show_centers=False):
    fig, axes = plt.subplots(1, len(results), figsize=(18, 3.4))
    for ax, result in zip(axes, results):
        Z = project_2d(result["X"])
        ax.scatter(Z[:, 0], Z[:, 1], c=result["labels"], s=16, cmap="tab10", alpha=0.82)
        if show_centers and result.get("centers") is not None:
            centers = np.asarray(result["centers"])
            if len(centers) > 0:
                centers_2d = centers if centers.shape[1] == 2 else PCA(n_components=2, random_state=SEED).fit(result["X"]).transform(centers)
                ax.scatter(centers_2d[:, 0], centers_2d[:, 1], c="black", marker="x", s=70)
        ax.set_title(f"D{result['rung']} ARI={result['ari']:.2f}")
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title)
    plt.show()


def plot_ari_curve(results, title):
    xs = [result["rung"] for result in results]
    ys = [result["ari"] for result in results]
    plt.figure(figsize=(6, 3.5))
    plt.plot(xs, ys, marker="o")
    plt.ylim(-0.05, 1.05)
    plt.xlabel("D1 to D5 ladder rung")
    plt.ylabel("Adjusted Rand Index")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()


## 3. The concept, built once on D1

The lesson objective is $$J=\sum_{i=1}^{m}\min_k \lVert x_i-\mu_k\rVert_2^2$$. We first audit the exact toy distances: a point $x=[1,2]$ is closer to $\mu_1=[1,1.5]$ than to $\mu_2=[4.5,4]$.

In [ ]:
# Formula audit: $J = \sum_i \min_k ||x_i - \mu_k||_2^2$
x = np.array([1.0, 2.0])
mu1 = np.array([1.0, 1.5])
mu2 = np.array([4.5, 4.0])
d1 = float(np.sum((x - mu1) ** 2))
d2 = float(np.sum((x - mu2) ** 2))
J = 4 * d1
assert d1 == 0.25
assert d2 == 16.25
assert J == 1.00
print(d1, d2, J)

Now implement the real k-means loop: deterministic k-means++ seeding, nearest-centroid assignment, centroid update, and inertia tracking.

In [ ]:
def kmeans_plus_plus_init(X, k, seed=SEED):
    local_rng = np.random.default_rng(seed)
    centers = [X[local_rng.integers(0, len(X))]]
    for _ in range(1, k):
        dist2 = np.min(pairwise_distances(X, np.vstack(centers), metric="sqeuclidean"), axis=1)
        probs = dist2 / dist2.sum()
        next_idx = local_rng.choice(len(X), p=probs)
        centers.append(X[next_idx])
    return np.vstack(centers)


def method(X, k, init="kmeans++", max_iter=80, seed=SEED):
    X = np.asarray(X, dtype=float)
    if init == "kmeans++":
        centers = kmeans_plus_plus_init(X, k, seed=seed)
    else:
        centers = X[:k].copy()
    for _ in range(max_iter):
        distances = pairwise_distances(X, centers, metric="sqeuclidean")
        labels = np.argmin(distances, axis=1)
        new_centers = centers.copy()
        for cluster_id in range(k):
            mask = labels == cluster_id
            if np.any(mask):
                new_centers[cluster_id] = X[mask].mean(axis=0)
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    inertia = float(np.sum(np.min(pairwise_distances(X, centers, metric="sqeuclidean"), axis=1)))
    return labels, centers, inertia

## 4. The dataset ladder

The same clustering function runs on D1 hand points, D2 clean blobs, D3 anisotropic overlap, D4 real Iris, and D5 real digits. Hidden labels are kept only for ARI scoring; the clustering method never receives them.

In [ ]:
rungs = cluster_ladder()
for row in preview_ladder(rungs):
    print(row)

## 5. Run the same method across D1-D5

The metric is Adjusted Rand Index against hidden generating labels. The labels are never passed into `method`; they are used only after clustering for evaluation.

In [ ]:
results = []
for rung, (name, X, y_true, k) in enumerate(rungs, start=1):
    X_use = StandardScaler().fit_transform(X)
    labels, centers, inertia = method(X_use, k)
    score = ari_score(y_true, labels)
    sil = safe_silhouette(X_use, labels)
    results.append({"rung": rung, "name": name, "X": X_use, "labels": labels, "centers": centers, "ari": score, "silhouette": sil, "inertia": inertia})
    print(f"D{rung} {name}: ARI={score:.3f} silhouette={sil:.3f} inertia={inertia:.2f}")

## 6. Results visualization

Each panel shows assignments on a two-dimensional view. The curve tracks ARI from D1 to D5.

In [ ]:
plot_cluster_panels(results, "k-means++ cluster assignments", show_centers=True)
plot_ari_curve(results, "k-means++ ARI vs ladder rung")

## 7. Pitfall on D5: unscaled pixel features dominate

The lesson warning says distances only see the numeric scale we provide. On D5, we create a bad copy where one pixel coordinate is offset and rerun k-means; then we fix it with scaling and seed-stability reruns.

In [ ]:
name, X5, y5, k5 = rungs[-1]
X_bad = X5.copy()
X_bad[:, 0] = X_bad[:, 0] * 100.0 + 50.0
bad_labels, bad_centers, bad_inertia = method(X_bad, k5)
bad_ari = ari_score(y5, bad_labels)
X_fixed = StandardScaler().fit_transform(X_bad)
fixed_scores = []
for seed in [3, 7, 11, 17]:
    labels, centers, inertia = method(X_fixed, k5, seed=seed)
    fixed_scores.append(ari_score(y5, labels))
print("bad unscaled ARI", round(bad_ari, 3))
print("scaled stability ARI", [round(score, 3) for score in fixed_scores])
print("scaled mean ARI", round(float(np.mean(fixed_scores)), 3))

## 8. Evaluate it + practice

- Metric: Adjusted Rand Index vs hidden labels, plus a no-skill sanity check from random labels.
- Sanity check: rerun with a nearby seed or hyperparameter and confirm the D5 story does not flip.
- Ablation: turn off the key idea in this lesson and watch ARI or stability drop.
- Failure signal: one cluster, all noise, or many singleton clusters usually means scale or hyperparameters dominate.
- D5 check: digits are real 64-D images, so visualization is a projection while clustering uses the full feature matrix.

Ablation: set `init="first"` in `method` and compare it with k-means++ on D2 and D5.

Practice prompts:
1. Change one hyperparameter on D3 and explain whether ARI or the assignment panel changes first.

2. Add a stability rerun on D5 and compare the resulting ARI values.

3. Replace StandardScaler with no scaling on D4 or D5 and describe the failure mode.